# 5장 실습 ① — 학습률과 옵티마이저

**PyTorch 판**

소용돌이 데이터로 학습률과 옵티마이저를 바꿔 가며 돌려 봅니다.

> **이 노트북에서 판마다 다른 셀은 `train_model()` 하나뿐입니다.**
> 실험을 돌리는 바깥 루프는 세 판이 글자까지 같습니다.
> 3장에서 한 이야기 — *프레임워크가 바뀌어도 딥러닝은 바뀌지 않는다* — 를
> 여기서 한 번 더 확인하게 됩니다.

## 5.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 5.1 실험대 — 소용돌이

이 문제의 **바닥**을 먼저 재 둡니다. 직선 하나로 최대 얼마나 맞히는가.
앞으로 나오는 숫자가 이 값 근처면 **"사실상 직선 하나"**라는 뜻입니다.

In [ ]:
# 소용돌이 — 하이퍼파라미터가 결과를 실제로 바꾸는 문제.
# 사과 데이터는 무엇을 해도 0.95가 나와서 이 장의 실험대가 될 수 없다.
x, y = data.spirals(n=1600, seed=42)
s = data.split(x, y, val_ratio=0.2, test_ratio=0.2, seed=42)
print(s.summary())

plot.scatter2d(s.x_train, s.y_train, class_names=("무리 0", "무리 1"),
               xlabel="$x_1$", ylabel="$x_2$", title="소용돌이 — 이 장의 실험대")
plt.show()

# 이 문제의 바닥: 직선 하나로 최대 얼마나 맞히는가
rng = np.random.default_rng(0)
best = 0.0
for _ in range(4000):
    w, b = rng.normal(size=2), rng.normal() * 3
    a = metrics.accuracy(y, (x @ w + b > 0).astype(int))
    best = max(best, a, 1 - a)
dlbook.record("ch05_spiral_single_line_acc", best)
print("→ 0.7 근처의 결과가 나오면 '사실상 직선 하나'라는 뜻입니다.")

## 5.2 학습 함수 — 여기만 판마다 다릅니다

아래 셀 하나가 이 판의 방식으로 모델을 만들고 학습시킵니다.
**이 아래의 모든 셀은 세 판이 같습니다.**

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

dlbook.set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

_ACT = {"relu": nn.ReLU, "sigmoid": nn.Sigmoid, "tanh": nn.Tanh}

def _init_(m, how):
    if not isinstance(m, nn.Linear):
        return
    if how == "zeros":
        nn.init.zeros_(m.weight)
    elif how == "random_normal":
        nn.init.normal_(m.weight, 0.0, 0.05)
    elif how == "he_normal":
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
    else:                                    # glorot_uniform = Xavier
        nn.init.xavier_uniform_(m.weight)
    nn.init.zeros_(m.bias)

def train_model(depth=3, units=32, act="relu", init="glorot_uniform",
                lr=0.01, opt="adam", epochs=60, bs=32, seed=42):
    """모델을 만들어 학습시키고 (시험 정확도, history)를 돌려준다.

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    dlbook.set_seed(seed)
    layers_ = []
    prev = 2
    for _ in range(depth):
        layers_ += [nn.Linear(prev, units), _ACT[act]()]
        prev = units
    layers_ += [nn.Linear(prev, 1)]          # 시그모이드는 손실 함수 안에 있다
    model = nn.Sequential(*layers_).to(device)
    model.apply(lambda m: _init_(m, init))

    criterion = nn.BCEWithLogitsLoss()
    optimizer = {"adam": torch.optim.Adam,
                 "sgd": torch.optim.SGD,
                 "rmsprop": torch.optim.RMSprop}[opt](model.parameters(), lr=lr)

    ds = TensorDataset(torch.tensor(s.x_train, dtype=torch.float32),
                       torch.tensor(s.y_train, dtype=torch.float32).view(-1, 1))
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    xv = torch.tensor(s.x_val, dtype=torch.float32).to(device)
    yv = torch.tensor(s.y_val, dtype=torch.float32).view(-1, 1).to(device)

    history = {"loss": [], "val_loss": []}
    for _ in range(dlbook.smoke.epochs(epochs)):
        model.train()
        losses = []
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            history["val_loss"].append(criterion(model(xv), yv).item())
        history["loss"].append(float(np.mean(losses)))

    model.eval()
    with torch.no_grad():
        logit = model(torch.tensor(s.x_test, dtype=torch.float32).to(device))
    pred = (logit.cpu().numpy().reshape(-1) > 0).astype("int64")
    return metrics.accuracy(s.y_test, pred), history

## 5.3 학습률을 바꿔 봅니다

은닉층 3개(각 32노드), ReLU, Adam, 60 epoch. **학습률만 바꿉니다.**

In [ ]:
rates = [0.0001, 0.001, 0.01, 0.1, 1.0]
results, curves = {}, {}

for lr in rates:
    acc, h = train_model(lr=lr)
    results[lr], curves[lr] = acc, h
    print(f"lr={lr:<8} 시험 정확도 {acc:.3f}   최종 학습 손실 {h['loss'][-1]:.4f}")

dlbook.record("ch05_lr_too_small_acc", results[0.0001])
dlbook.record("ch05_lr_good_acc",      results[0.01])
dlbook.record("ch05_lr_diverged_acc",  results[1.0])

## 5.4 손실 곡선으로 보기

숫자 100줄보다 곡선 하나가 낫습니다. 실무에서 학습을 지켜보는 방식입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for lr in rates:
    ax.plot(curves[lr]["loss"], label=f"lr = {lr}")
ax.set_xlabel("epoch"); ax.set_ylabel("학습 손실")
ax.set_title("학습률에 따른 손실 곡선"); ax.grid(alpha=0.3); ax.legend()
plt.show()

# 5.5절의 다섯 가지 모양 중 어디에 해당하는지 각 곡선을 분류해 보십시오.

## 5.5 옵티마이저를 바꿔 봅니다

같은 학습률(0.01), 같은 모델. **옵티마이저만 바꿉니다.**

In [ ]:
for opt in ("sgd", "rmsprop", "adam"):
    acc, h = train_model(lr=0.01, opt=opt)
    print(f"{opt:<9} 시험 정확도 {acc:.3f}   최종 학습 손실 {h['loss'][-1]:.4f}")
    dlbook.record(f"ch05_opt_{opt}_acc", acc)

## 정리

- 학습률은 **넓은 구간에서 아무 차이가 없다가, 그 구간을 벗어나면 완전히
  망합니다.** 정확한 값을 찾는 것이 아니라 **망하는 구간을 피하는 것**입니다.
- **Adam으로 시작하십시오.** SGD로 같은 결과를 내려면 학습률을 잘 골라야 합니다.

### 연습

1. 학습률 0.0001에서 epoch을 300으로 늘리면 정확도가 회복됩니까.
   회복된다면, 그것이 뜻하는 바는 무엇입니까.
2. SGD에 학습률 0.5를 주면 어떻게 됩니까. Adam과 견주어 보십시오.
3. `ReduceLROnPlateau` 콜백을 붙여 학습률 감쇠를 걸어 보십시오.